# Deep Learning 043 — Padding and Strides

Measured in the lesson: on a 5×5 image with a 3×3 filter, a **corner pixel takes
part in 1 convolution and the centre pixel in 9** — a 9× bias that is pure
geometry. Padding takes the corner to 4 and the bias to about 2×.

| Part | What we check |
|---|---|
| A | count how often each pixel is used |
| B | the same count with padding |
| C | the output-size formula, and why filters are odd |
| D | stride |

In [ ]:
import numpy as np

def usage_counts(n, f, p=0, s=1):
    """How many times each pixel of an n x n image is covered by the filter."""
    size = n + 2 * p
    counts = np.zeros((size, size), dtype=int)
    out = (size - f) // s + 1
    for i in range(0, out * s, s):
        for j in range(0, out * s, s):
            counts[i:i + f, j:j + f] += 1
    return counts[p:p + n, p:p + n] if p else counts

## Part A — No padding

In [ ]:
c = usage_counts(5, 3)
print(c)
print(f"\ncorner {c[0,0]}, edge {c[0,2]}, centre {c[2,2]}  ->  bias {c[2,2] / c[0,0]:.0f}x")
assert c[0, 0] == 1 and c[2, 2] == 9

The centre of the image is examined nine times as often as the corner. That is
not about the picture — it is about where a 3×3 window can sit.

## Part B — With padding

In [ ]:
cp = usage_counts(5, 3, p=1)
print(cp)
print(f"\ncorner {cp[0,0]}, centre {cp[2,2]}  ->  bias {cp[2,2] / cp[0,0]:.2f}x")
assert cp[0, 0] == 4
print("\nThe corner went from 1 use to 4, and the bias from 9x to about 2x.")

## Part C — Output size

In [ ]:
def out_size(n, f, p=0, s=1):
    return (n + 2 * p - f) // s + 1

print(f"{'n':>5}{'f':>4}{'p':>4}{'out':>6}   note")
for n, f, p in ((6, 3, 0), (6, 3, 1), (28, 5, 0), (28, 5, 2), (28, 4, 1)):
    note = "same size" if out_size(n, f, p) == n else ""
    print(f"{n:>5}{f:>4}{p:>4}{out_size(n, f, p):>6}   {note}")

# p = (f-1)/2 preserves the size - and only works when f is odd
for f in (3, 5, 7):
    p = (f - 1) // 2
    assert out_size(28, f, p) == 28
print("\np = (f-1)/2 keeps the size, and it is a whole number only for odd f.")
print("That is why filters are 3x3, 5x5, 7x7 and almost never 4x4.")

## Part D — Stride

In [ ]:
print(f"{'stride':>7}{'output':>8}   from a 28x28 image, 3x3 filter, p=1")
for s in (1, 2, 3):
    print(f"{s:>7}{out_size(28, 3, p=1, s=s):>8}")
assert out_size(28, 3, p=1, s=2) == 14
print("\ns=2 roughly halves each dimension, so it quarters the number of values.")

c2 = usage_counts(5, 3, p=0, s=2)
print("\nusage counts at stride 2 (some pixels are now skipped entirely):\n", c2)

## What to take away

- Without padding, **a corner pixel is convolved once and the centre nine times**
  (5×5, 3×3 filter) — a 9× bias, and the output shrinks every layer.
- **Padding** fixes both: the corner goes to 4 uses, the bias to about 2×.
- **n_out = n + 2p − f + 1**, and **p = (f−1)/2** preserves the size — which only
  divides evenly for odd f.
- **Stride** is how far the filter jumps; s = 2 roughly halves each dimension.

## Exercises

1. Compute the bias for a 7×7 image with a 3×3 filter, with and without padding.
2. What padding keeps a 32×32 image the same size with a 7×7 filter?
3. At stride 3 on a 5×5 image, which pixels are never covered at all?